# QA Prompt Tuning Notebook

逐类调试出题提示词。每次聚焦一类题型，反复调 prompt 直到输出质量满意，然后将定稿抄回 `prompts.py`。

In [ ]:
import os, sys, json, re, time
sys.path.append(os.path.dirname(os.getcwd()))

from openai import OpenAI
from config import API_BASE, API_KEY, MODEL, MAX_TOKENS_ANSWER
from pipeline import sliding_window_chunks, get_book_content

client = OpenAI(base_url=API_BASE, api_key=API_KEY, timeout=600.0)

In [ ]:
# ── 选择测试用的书 ──
BOOK_NAME = "43.cleaned.txt"  # 改成你想用的书名
BOOK_PATH = os.path.join(os.path.dirname(os.getcwd()), "books", "train", BOOK_NAME)

content = get_book_content(BOOK_NAME)
windows = sliding_window_chunks(content)
print(f"{BOOK_NAME}: {len(windows)} windows, {len(content)} chars")
print(f"Window size: ~{len(windows[0]['text'])} chars each")

---
## 快速生成 summaries

这里用一个轻量版的 summarize 跑一遍，产出各 chunk 的 summary。
跑过一次后可以缓存，下次直接跳过。

In [ ]:
CACHE_FILE = f".cache_{BOOK_NAME.replace('.','_')}_summaries.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        cache = json.load(f)
    summaries = cache["summaries"]
    print(f"Loaded {len(summaries)} summaries from cache.")
else:
    from prompts import SUMMARIZE_CHUNK_PROMPT
    summaries = []
    for i, w in enumerate(windows):
        msgs = [
            {"role": "system", "content": SUMMARIZE_CHUNK_PROMPT},
            {"role": "user", "content": f"[{BOOK_NAME} | {i+1}/{len(windows)}]\n{w['text']}"},
        ]
        resp = client.chat.completions.create(
            model=MODEL,
            messages=msgs,
            temperature=0.3,
            max_tokens=2048,
        )
        s = resp.choices[0].message.content or ""
        summaries.append(s.strip())
        print(f"  [{i+1}/{len(windows)}] done", flush=True)
    
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump({"summaries": summaries}, f, ensure_ascii=False)
    print(f"Saved {len(summaries)} summaries to cache.")

---
## 拼完整的 summaries_text（带 Section 编号）

和 pipeline.py 里 `process_books` 的拼接方式一致。

In [ ]:
summaries_text = "\n\n".join(
    f"[Section {i+1}]\n{s}" for i, s in enumerate(summaries)
)
print(f"Total: {len(summaries_text)} chars, {len(summaries)} sections")

---
## 题型选择 & Prompt 调试区

下面的 prompt 模板可以任意编辑后重新运行。选一类题，改 prompt，点运行看效果。

In [ ]:
# ── 当前调试的题型 ──
QUESTION_TYPE = "relationship"  # 可选: default, relationship, timeline, causal, detail, global

# ── 题型说明（仅展示） ──
TYPE_DESCRIPTION = {
    "default": "通用题型 — 混合出各类题",
    "relationship": "人物关系 — 角色之间的社会关系、情感关系、家庭关系",
    "timeline": "剧情时序 — 事件发生的先后顺序、因果关系",
    "causal": "因果推理 — 角色行为背后的动机和原因",
    "detail": "细节定位 — 特定事实的精确查找（年龄、地点、身份等）",
    "global": "全局综合 — 需要跨段落整合的综合性问题",
}
print(f"当前题型: {QUESTION_TYPE} — {TYPE_DESCRIPTION.get(QUESTION_TYPE, '')}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# TODO: 在这里修改 prompt，反复调试
# ════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """You are a QA generation expert specializing in character relationship questions.

Based on the section summaries below, generate 3-5 high-quality question-answer pairs
about character RELATIONSHIPS — romantic, familial, social, or professional.

Requirements:
- Each question must involve at least TWO named characters.
- Focus on: who loves whom, who is married to whom, who works for whom,
  who is related to whom, who is rivals with whom.
- Questions should be open-ended (not yes/no).
- Answers should be accurate and cite [N] section numbers for each factual claim.
- Questions must NOT contain section numbers.

Output format — a JSON array:
[
    {
        "book": "<book title>",
        "question": "<question>",
        "answer": "<answer with [N] citations>"
    },
    ...
]

Return ONLY the JSON array, no other text."""

TEMPERATURE = 0.3

---
## 跑一次看效果

In [ ]:
book_titles = {BOOK_NAME: "Test Book"}  # 简化版
user_content = f"Books: Test Book\n\nSummaries:\n\n{summaries_text}"

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ],
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS_ANSWER,
)

raw_output = resp.choices[0].message.content or ""
print(raw_output)

In [ ]:
# ── 格式化展示结果 ──
def _strip_code_fence(text: str) -> str:
    s = re.sub(r'^```(?:json)?\s*\n?', '', text.strip(), flags=re.IGNORECASE)
    s = re.sub(r'\n?```\s*$', '', s, flags=re.IGNORECASE)
    return s.strip()

cleaned = _strip_code_fence(raw_output)
try:
    qa_list = json.loads(cleaned)
    if not isinstance(qa_list, list):
        qa_list = [qa_list]
    print(f"\n=== 生成 {len(qa_list)} 道 {TYPE_DESCRIPTION.get(QUESTION_TYPE, QUESTION_TYPE)} 题 ===\n")
    for i, qa in enumerate(qa_list, 1):
        print(f"--- Q{i} ---")
        print(f"Q: {qa.get('question', '')}")
        print(f"A: {qa.get('answer', '')[:200]}..." if len(qa.get('answer', '')) > 200 else f"A: {qa.get('answer', '')}")
        print()
except json.JSONDecodeError as e:
    print(f"JSON 解析失败: {e}")
    print("原始输出:", raw_output[:500])

---
## 人工评分（可选）

看一遍生成的题，给个分，记下改进了什么。

In [ ]:
# 满意了就复刻到 prompts.py 里的 QUESTION_TYPES 对应条目
print("评分标准:")
print("  5 — 完美，可以直接用")
print("  4 — 质量高，小改就行")
print("  3 — 一般，需要明显改进")
print("  2 — 差，需要重写")
print("  1 — 完全跑偏")

SCORE = 3  # 手动改
NOTES = "问题有的太简单了，需要增加复杂度"  # 手动改
print(f"\n评分: {SCORE}/5")
print(f"备注: {NOTES}")

---
## 调好的 prompt 抄回 prompts.py

满意后，把上面 SYSTEM_PROMPT 的内容复制到 `prompts.py` 的 `QUESTION_TYPES` 字典中对应条目。

```python
# 在 prompts.py 中添加：
QUESTION_TYPES = {
    "relationship": {
        "label": "人物关系",
        "system_prompt": """你粘贴的内容""",
        "temperature": 0.3,
        "max_tokens": 4096,
        "num_questions": "3-5",
    },
}
```